## Preprocessing and features engineering

Now we have a clean dataset with tennis matches starting from **2005-07-04** until **2026** and we need to preprocess it, by reducing the number of features and creating new ones to improve the model performance. Afterwards, we will split the dataset into training and test sets and start the model training.

In [162]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEANED_PATH = Path("../data/cleaned/tennis_matches.xlsx")

df_raw = pd.read_excel(CLEANED_PATH)
df_raw.head()

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,WRank,LRank,WPts,LPts,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5,B365W,B365L,Wsets,Lsets
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,robredo t.,tabara m.,20.0,112.0,1425.0,381.0,7.0,5.0,6.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00,2.0,0.0
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,vinciguerra a.,ryderstedt m.,917.0,132.0,9.0,326.0,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66,2.0,0.0
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,verdasco f.,pospisil j.,59.0,432.0,640.0,64.0,6.0,2.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,ginepri r.,oudsema s.,103.0,693.0,405.0,22.0,6.0,2.0,6.0,7.0,6.0,0.0,NaN,NaN,NaN,NaN,1.12,5.50,2.0,1.0
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,spadea v.,popp a.,49.0,176.0,730.0,246.0,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN,2.50,1.50,2.0,1.0


Firstly, we must randomly swap the winner and loser players in each match, to avoid the model learning about choosing always the first player as the winner, causing a data leakage.

In [163]:
df_prepared = df_raw.copy()

# Randomly assign the winner to Player 1 or Player 2.
rng = np.random.default_rng(42)
winner_is_player_1 = rng.random(len(df_prepared)) < 0.5

df_prepared["Player_1"] = np.where(
    winner_is_player_1, df_raw["Winner"], df_raw["Loser"]
)
df_prepared["Player_2"] = np.where(
    winner_is_player_1, df_raw["Loser"], df_raw["Winner"]
)

# Target: 1 if Player 1 wins, otherwise 0.
df_prepared["y"] = winner_is_player_1.astype(int)

# Align player statistics with the randomized player positions.
for name, winner_column, loser_column in [
    ("Rank", "WRank", "LRank"),
    ("Pts", "WPts", "LPts"),
    ("Odds", "B365W", "B365L"),
    ("Sets", "Wsets", "Lsets"),
    ("games1", "W1", "L1"),
    ("games2", "W2", "L2"),
    ("games3", "W3", "L3"),
    ("games4", "W4", "L4"),
    ("games5", "W5", "L5"),
]:
    df_prepared[f"{name}_1"] = np.where(
        winner_is_player_1,
        df_raw[winner_column],
        df_raw[loser_column],
    )
    df_prepared[f"{name}_2"] = np.where(
        winner_is_player_1,
        df_raw[loser_column],
        df_raw[winner_column],
    )
    
df_prepared.head()

,Date,Series,Court,Surface,Round,Tournament,Location,Best of,Winner,Loser,WRank,LRank,WPts,LPts,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5,B365W,B365L,Wsets,Lsets,Player_1,Player_2,y,Rank_1,Rank_2,Pts_1,Pts_2,Odds_1,Odds_2,Sets_1,Sets_2,games1_1,games1_2,games2_1,games2_2,games3_1,games3_2,games4_1,games4_2,games5_1,games5_2
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,robredo t.,tabara m.,20.0,112.0,1425.0,381.0,7.0,5.0,6.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00,2.0,0.0,tabara m.,robredo t.,0,112.0,20.0,381.0,1425.0,6.0,1.10,0.0,2.0,5.0,7.0,0.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,Bastad,3.0,vinciguerra a.,ryderstedt m.,917.0,132.0,9.0,326.0,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66,2.0,0.0,vinciguerra a.,ryderstedt m.,1,917.0,132.0,9.0,326.0,2.1,1.66,2.0,0.0,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,Gstaad,3.0,verdasco f.,pospisil j.,59.0,432.0,640.0,64.0,6.0,2.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,pospisil j.,verdasco f.,0,432.0,59.0,64.0,640.0,NaN,NaN,0.0,2.0,2.0,6.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,ginepri r.,oudsema s.,103.0,693.0,405.0,22.0,6.0,2.0,6.0,7.0,6.0,0.0,NaN,NaN,NaN,NaN,1.12,5.50,2.0,1.0,oudsema s.,ginepri r.,0,693.0,103.0,22.0,405.0,5.5,1.12,1.0,2.0,2.0,6.0,7.0,6.0,0.0,6.0,NaN,NaN,NaN,NaN
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,Newport,3.0,spadea v.,popp a.,49.0,176.0,730.0,246.0,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN,2.50,1.50,2.0,1.0,spadea v.,popp a.,1,49.0,176.0,730.0,246.0,2.5,1.50,2.0,1.0,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN


#### Numerical features

We can start by creating simple engineered features, such as the differences between the players' ranks, points, and betting market odds.

We can also use logarithmic differences. This is useful because these variables are not linear.

**Example 1**

- Rank 1 vs 10
- Rank 101 vs 110

The absolute difference is the same (9), but the first difference is much more meaningful than the second one.

**Example 2**

- Points 9000 vs 8000
- Points 1200 vs 1000

Same considerations as example 1.

**Example 3**

- Odds 1.20 vs 1.50
- Odds 4.00 vs 4.30

Same absolute difference, but the second one is much more uncertain than the first one.

For betting odds, we use a log-ratio instead of a raw difference. This is closer to the market implied probability: positive values mean Player 1 is favored by the market, negative values mean Player 2 is favored.

In [164]:
p1_rank, p2_rank = df_prepared["Rank_1"], df_prepared["Rank_2"]
p1_pts, p2_pts = df_prepared["Pts_1"], df_prepared["Pts_2"]
p1_odds, p2_odds = df_prepared["Odds_1"], df_prepared["Odds_2"]

# Standard differences

df_prepared["Rank_Diff"] = p1_rank - p2_rank
df_prepared["Points_Diff"] = p1_pts - p2_pts

# Logarithmic ratios
df_prepared["Rank_Log_Ratio"] = np.log(p2_rank) - np.log(p1_rank)
df_prepared["Points_Log_Ratio"] = np.log1p(p1_pts) - np.log1p(p2_pts)


# Market log-odds ratio: positive means Player 1 is favored by the market.
# This is more informative than a raw odds difference because betting odds are multiplicative.
df_prepared["Odds_Diff"] = p1_odds - p2_odds
valid_odds = (p1_odds > 0) & (p2_odds > 0)
df_prepared["Odds_Log_Ratio"] = np.nan
df_prepared.loc[valid_odds, "Odds_Log_Ratio"] = np.log(
    p2_odds[valid_odds] / p1_odds[valid_odds]
)

In [165]:
numerical_features = [
    "Rank_Diff", "Points_Diff", "Odds_Diff",
    "Rank_Log_Ratio", "Points_Log_Ratio", 
    "Odds_Log_Ratio",
]

We can convert **Best of** feature into a binary feature **Best of 5**, which is 1 if the match is best of 5 sets and 0 otherwise.

In [166]:
df_prepared["Best_of_5"] = (df_raw["Best of"] == 5).astype(int)
numerical_features += ["Best_of_5"]

We use the cleaned players dataset to create an age-difference feature.

The players file already contains match-compatible keys such as `Ruud C.` and keeps duplicated keys.
Here we only select a date of birth when exactly one candidate gives a plausible age at match date.


In [167]:
PLAYERS_PATH = Path("../data/cleaned/tennis_players.xlsx")
players_df = pd.read_excel(PLAYERS_PATH)

MIN_PLAYER_AGE, MAX_PLAYER_AGE = 16, 45
dob_candidates = (
    players_df.dropna(subset=["player_key", "dob"])[["player_key", "dob"]]
    .drop_duplicates(subset=["player_key", "dob"])
)

def compute_age(df, player_col):
    left = df[["Date", player_col]].reset_index() # create new index column to preserve original row order
    merged = left.merge(dob_candidates, left_on=player_col, right_on="player_key", how="left") # left join
    merged["age"] = (merged["Date"] - merged["dob"]).dt.days / 365.25
    plausible = merged[merged["age"].between(MIN_PLAYER_AGE, MAX_PLAYER_AGE)] # keep only plausible ages
    n_candidates = plausible.groupby("index")["age"].transform("count") # count column with number of candidates for each row
    unique_age = plausible[n_candidates == 1].set_index("index")["age"] # keep only rows with count == 1
    return unique_age.reindex(df.index)

age_1 = compute_age(df_prepared, "Player_1")
age_2 = compute_age(df_prepared, "Player_2")
df_prepared["Age_Diff"] = age_1 - age_2

df_prepared["Age_1"] = age_1.round(2)
df_prepared["Age_2"] = age_2.round(2)

print(df_prepared[["Player_1", "Player_2", "Age_1", "Age_2", "Age_Diff", "Date"]].head(10))

coverage = df_prepared["Age_Diff"].notna().sum() / len(df_prepared)
print(f"Age-difference matchups: {coverage * 100:.2f} %")

         Player_1       Player_2  Age_1  Age_2  Age_Diff       Date
0       tabara m.     robredo t.  25.90  23.18  2.721424 2005-07-04
1  vinciguerra a.  ryderstedt m.    NaN  20.64       NaN 2005-07-04
2     pospisil j.    verdasco f.  24.40  21.63  2.762491 2005-07-04
3      oudsema s.     ginepri r.  19.01  22.74 -3.731691 2005-07-04
4       spadea v.        popp a.  30.96  28.66  2.297057 2005-07-04
5  burgsmuller l.    lapentti g.  29.58  22.44  7.137577 2005-07-04
6      haehnel j.     youzhny m.  24.97  23.03  1.946612 2005-07-05
7       dlouhy l.   ferrero j.c.  22.24  25.39 -3.154004 2005-07-05
8      berdych t.         kim k.  19.80    NaN       NaN 2005-07-05
9      almagro n.     acasuso j.  19.87  22.71 -2.836413 2005-07-05
Age-difference matchups: 77.38 %


### Hand matchup

In [168]:
hand_data = players_df.dropna(subset=["player_key", "hand"]).groupby("player_key")["hand"]

player_hand = hand_data.first().where(hand_data.nunique() == 1).to_dict()

def get_hand(player_name):
    return player_hand.get(player_name, np.nan)

hand_score = {"L": 1, "R": -1}

hand_1 = df_prepared["Player_1"].map(get_hand)
hand_2 = df_prepared["Player_2"].map(get_hand)

h1, h2 = hand_1.map(hand_score), hand_2.map(hand_score)

both_known = hand_1.isin(["L", "R"]) & hand_2.isin(["L", "R"])

df_prepared["Hand_Advantage"] = np.where(both_known, (h1 - h2) / 2, np.nan)
numerical_features += ["Hand_Advantage"]

df_prepared["Hand_1"] = hand_1
df_prepared["Hand_2"] = hand_2
print(df_prepared[["Player_1", "Player_2", "Hand_1", "Hand_2", "Hand_Advantage"]].head(10))


coverage = df_prepared["Hand_Advantage"].notna().sum() / len(df_prepared)
print(f"Hand advantage matchups: {coverage * 100:.2f} %")

         Player_1       Player_2 Hand_1 Hand_2  Hand_Advantage
0       tabara m.     robredo t.      R      R             0.0
1  vinciguerra a.  ryderstedt m.    NaN      L             NaN
2     pospisil j.    verdasco f.      R      L            -1.0
3      oudsema s.     ginepri r.      R      R             0.0
4       spadea v.        popp a.      R      R             0.0
5  burgsmuller l.    lapentti g.      R      R             0.0
6      haehnel j.     youzhny m.      R      R             0.0
7       dlouhy l.   ferrero j.c.      R      R             0.0
8      berdych t.         kim k.      R    NaN             NaN
9      almagro n.     acasuso j.      R      R             0.0
Hand advantage matchups: 76.23 %


### Home factor

A player performs better when playing in his home country.

In [169]:
# All cities
print(sorted(df_raw["Location"].unique()))

LOCATION_TO_IOC = {
    "'s-Hertogenbosch": "NED", "Amersfoort": "NED", "Rotterdam": "NED",
    "Acapulco": "MEX", "Los Cabos": "MEX",
    "Adelaide": "AUS", "Brisbane": "AUS", "Melbourne": "AUS", "Sydney": "AUS",
    "Almaty": "KAZ", "Nur-Sultan": "KAZ",
    "Antalya": "TUR", "Istanbul": "TUR",
    "Antwerp": "BEL", "Brussels": "BEL",
    "Athens": "GRE",
    "Atlanta": "USA", "Cincinnati": "USA", "Dallas": "USA", "Delray Beach": "USA",
    "Houston": "USA", "Indian Wells": "USA", "Indianapolis": "USA", "Las Vegas": "USA",
    "Los Angeles": "USA", "Memphis": "USA", "Miami": "USA", "New Haven": "USA",
    "New York": "USA", "Newport": "USA", "San Diego": "USA", "San Jose": "USA",
    "Washington": "USA", "Winston-Salem": "USA",
    "Auckland": "NZL",
    "Bangkok": "THA",
    "Banja Luka": "BIH",
    "Barcelona": "ESP", "Gijon": "ESP", "Madrid": "ESP", "Mallorca": "ESP",
    "Marbella": "ESP", "Valencia": "ESP",
    "Basel": "SUI", "Geneva": "SUI", "Gstaad": "SUI",
    "Bastad": "SWE", "Stockholm": "SWE",
    "Beijing": "CHN", "Chengdu": "CHN", "Hangzhou": "CHN", "Shanghai": "CHN",
    "Shenzhen": "CHN", "Zhuhai": "CHN",
    "Belgrade": "SRB",
    "Bogota": "COL",
    "Bucharest": "ROU",
    "Budapest": "HUN",
    "Buenos Aires": "ARG", "Cordoba": "ARG",
    "Cagliari": "ITA", "Florence": "ITA", "Napoli": "ITA", "Palermo": "ITA",
    "Parma": "ITA", "Rome": "ITA", "Sardinia": "ITA", "Turin": "ITA",
    "Casablanca": "MAR", "Marrakech": "MAR",
    "Chennai": "IND", "Mumbai": "IND", "Pune": "IND",
    "Cologne": "GER", "Dusseldorf": "GER", "Halle": "GER", "Hamburg": "GER",
    "Munich": "GER", "Stuttgart": "GER",
    "Costa Do Sauipe": "BRA", "Rio de Janeiro": "BRA", "Sao Paulo": "BRA",
    "Doha": "QAT",
    "Dubai": "UAE",
    "Eastbourne": "GBR", "London": "GBR", "Nottingham": "GBR", "Queens Club": "GBR",
    "Estoril": "POR", "Oeiras": "POR",
    "Ho Chi Min City": "VIE",
    "Hong Kong": "HKG",
    "Johannesburg": "RSA",
    "Kitzbuhel": "AUT", "Portschach": "AUT", "Vienna": "AUT",
    "Kuala Lumpur": "MAS",
    "Lyon": "FRA", "Marseille": "FRA", "Metz": "FRA", "Montpellier": "FRA",
    "Nice": "FRA", "Paris": "FRA",
    "Monte Carlo": "MON",
    "Montreal": "CAN", "Toronto": "CAN",
    "Moscow": "RUS", "St. Petersburg": "RUS",
    "Quito": "ECU",
    "Santiago": "CHI", "Vina del Mar": "CHI",
    "Seoul": "KOR",
    "Singapore": "SIN",
    "Sofia": "BUL",
    "Sopot": "POL", "Warsaw": "POL",
    "Tel Aviv": "ISR",
    "Tokyo": "JPN",
    "Umag": "CRO", "Zagreb": "CRO",
}

# some cities have end spaces (e.g. "Estoril ")
df_prepared["Location"] = df_prepared["Location"].str.strip()

# Map the location to IOC (e.g. "Rome" -> "ITA")
df_prepared["Location_IOC"] = df_prepared["Location"].map(LOCATION_TO_IOC)

# Get player IOC
ioc_data = players_df.dropna(subset=["player_key", "ioc"]).groupby("player_key")["ioc"]
player_ioc = ioc_data.first().where(ioc_data.nunique() == 1).to_dict()

df_prepared["Home_1"] = df_prepared["Player_1"].map(player_ioc)
df_prepared["Home_2"] = df_prepared["Player_2"].map(player_ioc)

home_1_flag = df_prepared["Home_1"] == df_prepared["Location_IOC"]
home_2_flag = df_prepared["Home_2"] == df_prepared["Location_IOC"]
both_known = df_prepared["Home_1"].notna() & df_prepared["Home_2"].notna() & df_prepared["Location_IOC"].notna()

# Compute the difference in home advantage between Player 1 and Player 2.
df_prepared["Home_Advantage"] = np.where(both_known, home_1_flag.astype(int) - home_2_flag.astype(int), np.nan)
numerical_features += ["Home_Advantage"]

print(df_prepared[["Player_1", "Player_2", "Location", "Home_1", "Home_2", "Home_Advantage"]].head(10))

coverage = df_prepared["Home_Advantage"].notna().sum() / len(df_prepared)
print(f"Home advantage matchups: {coverage * 100:.2f} %")

["'s-Hertogenbosch", 'Acapulco', 'Adelaide', 'Almaty', 'Amersfoort', 'Antalya', 'Antwerp', 'Athens', 'Atlanta', 'Auckland', 'Bangkok', 'Banja Luka', 'Barcelona', 'Basel', 'Bastad', 'Beijing', 'Belgrade', 'Bogota', 'Brisbane', 'Brussels', 'Bucharest', 'Budapest', 'Buenos Aires', 'Cagliari', 'Casablanca', 'Chengdu', 'Chennai', 'Cincinnati', 'Cologne', 'Cordoba', 'Costa Do Sauipe', 'Dallas', 'Delray Beach', 'Doha', 'Dubai', 'Dubai ', 'Dusseldorf', 'Eastbourne', 'Estoril', 'Estoril ', 'Florence', 'Geneva', 'Gijon', 'Gstaad', 'Halle', 'Hamburg', 'Hangzhou', 'Ho Chi Min City', 'Hong Kong', 'Houston', 'Indian Wells', 'Indianapolis', 'Istanbul', 'Johannesburg ', 'Kitzbuhel', 'Kuala Lumpur', 'Las Vegas', 'London', 'Los Angeles', 'Los Cabos', 'Lyon', 'Madrid', 'Mallorca', 'Marbella', 'Marrakech', 'Marseille', 'Melbourne', 'Memphis', 'Metz', 'Miami', 'Monte Carlo', 'Montpellier', 'Montreal', 'Moscow', 'Mumbai', 'Munich', 'Napoli', 'New Haven', 'New York', 'Newport', 'Nice', 'Nottingham', 'Nur-Sul

#### Advanced features

We can improve the model performance by creating *advanced* features about players ELO ratings, fatigue and head to head statistics.

Firstly, we sort the dataset by date to avoid data leakage

In [170]:
# Elo must be calculated in chronological order.
df_prepared["Date"] = pd.to_datetime(
    df_prepared["Date"],
    errors="raise",
)

# sort dataframe by date to avoid data leakage when calculating Elo ratings
df_prepared = (
    df_prepared
    .sort_values("Date", kind="stable")
    .reset_index(drop=True) # reset index after sorting
)

#### Fatigue

In [171]:
from collections import defaultdict, deque

FATIGUE_WINDOW_DAYS = 10
FATIGUE_DECAY_DAYS = 3.0
DEFAULT_REST_DAYS = 30

recent_matches = defaultdict(list)
fatigue_diff = []


def player_fatigue(player, match_date):
    """Return recent match load before the current match."""
    fatigue = 0.0

    for previous_date in recent_matches[player]:
        days_since_match = (match_date - previous_date).days

        if 0 < days_since_match <= FATIGUE_WINDOW_DAYS:
            fatigue += np.exp(-days_since_match / FATIGUE_DECAY_DAYS)

    return fatigue


def player_rest_days(player, match_date):
    """Return days since previous match, using a neutral value for new players."""
    if not recent_matches[player]:
        return DEFAULT_REST_DAYS
    return (match_date - recent_matches[player][-1]).days


def update_fatigue(player_1, player_2, match_date):
    """Store pre-match fatigue difference, then update recent match dates."""
    fatigue_1 = player_fatigue(player_1, match_date)
    fatigue_2 = player_fatigue(player_2, match_date)
    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    # Store pre-match fatigue difference to avoid data leakage.
    fatigue_diff.append(fatigue_1 - fatigue_2)

    # Update match history only after computing the feature.
    recent_matches[player_1].append(match_date)
    recent_matches[player_2].append(match_date)

    return rest_days_1, rest_days_2

#### ELO ratings

Players start with an Elo rating of 1500, and the expected win probability for Player 1 is computed as:

$$ E_1 = \frac{1}{1 + 10^{\frac{R_2 - R_1}{400}}} $$

Ratings are updated post-match via $\Delta R = K \cdot (S_1 - E_1)$, where:

- $R_1, R_2$ are the pre-match Elo ratings of Player 1 and Player 2
- $E_1$ is the expected probability that Player 1 wins
- $S_1$ is the actual result: 1 if Player 1 wins, 0 otherwise
- $K$ controls how strongly ratings are updated

The update factor $K$ dynamically decreases with career matches $M_i(t)$ to reflect growing rating stability:

$$ K_i(t) = \frac{250}{(M_i(t) + 5)^{0.4}} $$

Surface Elo is calculated using the same logic as standard Elo but is tracked independently for each surface. To leverage both, I use a "blended" rating, a 50/50 weighted average of the overall Elo and the surface-specific Elo. This approach stabilizes predictions for surfaces where a player has limited match data by anchoring their performance to their overall ability.

In [172]:
BASE_ELO = 1500.0
SURFACE_BLEND_WEIGHT = 0.5

elo_ratings = {}
surface_elo_ratings = {}
matches_played = {}
matches_played_surface = {}

elo_diff = []
surface_elo_diff = []

def dynamic_k(played):
    return 250 / ((played + 5) ** 0.4)

def blended_surface_rating(overall_rating, surface_rating):
    return (1 - SURFACE_BLEND_WEIGHT) * overall_rating + SURFACE_BLEND_WEIGHT * surface_rating

def expected_score(rating_1, rating_2):
    return 1 / (1 + 10 ** ((rating_2 - rating_1) / 400))

def update_elo_ratings(player_1, player_2, surface, y):
    elo_1 = elo_ratings.get(player_1, BASE_ELO)
    elo_2 = elo_ratings.get(player_2, BASE_ELO)
    surf_elo_1 = surface_elo_ratings.get((player_1, surface), BASE_ELO)
    surf_elo_2 = surface_elo_ratings.get((player_2, surface), BASE_ELO)

    blended_1 = blended_surface_rating(elo_1, surf_elo_1)
    blended_2 = blended_surface_rating(elo_2, surf_elo_2)

    expected_1 = expected_score(elo_1, elo_2)
    expected_surface_1 = expected_score(blended_1, blended_2)

    elo_diff.append(elo_1 - elo_2)
    surface_elo_diff.append(blended_1 - blended_2)

    k1 = dynamic_k(matches_played.get(player_1, 0))
    k2 = dynamic_k(matches_played.get(player_2, 0))
    elo_ratings[player_1] = elo_1 + k1 * (y - expected_1)
    elo_ratings[player_2] = elo_2 - k2 * (y - expected_1)

    k_surf_1 = dynamic_k(matches_played_surface.get((player_1, surface), 0))
    k_surf_2 = dynamic_k(matches_played_surface.get((player_2, surface), 0))
    surface_elo_ratings[(player_1, surface)] = surf_elo_1 + k_surf_1 * (y - expected_surface_1)
    surface_elo_ratings[(player_2, surface)] = surf_elo_2 - k_surf_2 * (y - expected_surface_1)

    matches_played[player_1] = matches_played.get(player_1, 0) + 1
    matches_played[player_2] = matches_played.get(player_2, 0) + 1
    matches_played_surface[(player_1, surface)] = matches_played_surface.get((player_1, surface), 0) + 1
    matches_played_surface[(player_2, surface)] = matches_played_surface.get((player_2, surface), 0) + 1

#### Recent form

We can store the last 5 matches played by each player using a queue and compute the **recent form** as the average of the last 5 matches played.

E.g. 
- $[1, 0, 1, 0, 1] \implies 0.6$

- $[1] \implies 1.0$

- $[0, 0, 1] \implies 0.33$

We can also try a weighted recent form, assigning an higher weight to most important matches as

$$ \texttt{weighted\_recent\_form} = \texttt{result} \cdot \frac{\texttt{opponent\_ELO}}{\texttt{BASE\_ELO}} $$

In [173]:
RECENT_FORM_WINDOW = 5
recent_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_form_diff = []

recent_surface_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_surface_form_diff = []

def update_recent_results(player_1, player_2, surface, y):
    form_1 = np.mean(recent_results[player_1]) if recent_results[player_1] else 0.5
    form_2 = np.mean(recent_results[player_2]) if recent_results[player_2] else 0.5
    form_surf_1 = np.mean(recent_surface_results[(player_1, surface)]) if recent_surface_results[(player_1, surface)] else 0.5
    form_surf_2 = np.mean(recent_surface_results[(player_2, surface)]) if recent_surface_results[(player_2, surface)] else 0.5

    recent_form_diff.append(form_1 - form_2)
    recent_surface_form_diff.append(form_surf_1 - form_surf_2)

    recent_results[player_1].append(y)
    recent_results[player_2].append(1 - y)
    recent_surface_results[(player_1, surface)].append(y)
    recent_surface_results[(player_2, surface)].append(1 - y)

We can also compute the **dominance form**, by considering the number of games won by each player in the last matches, instead of just the match result.

In [174]:
"""Compute number of games won by each player """
games_1 = df_prepared[["games1_1", "games2_1", "games3_1", "games4_1", "games5_1"]]
games_2 = df_prepared[["games1_2", "games2_2", "games3_2", "games4_2", "games5_2"]]

df_prepared["Games_P1"] = games_1.sum(axis=1, skipna=True)
df_prepared["Games_P2"] = games_2.sum(axis=1, skipna=True)

print(df_prepared[["Games_P1", "Games_P2", "Wsets", "Lsets", "y"]].head())

   Games_P1  Games_P2  Wsets  Lsets  y
0       5.0      13.0    2.0    0.0  0
1      12.0       4.0    2.0    0.0  1
2       6.0      12.0    2.0    0.0  0
3       9.0      18.0    2.0    1.0  0
4      16.0      11.0    2.0    1.0  1


In [175]:
DOMINANCE_FORM_WINDOW = 5
weighted_dominance_results = defaultdict(lambda: deque(maxlen=DOMINANCE_FORM_WINDOW))
dominance_form_diff = []

def update_dominance_form(player_1, player_2, games_won_1, games_total_1):

    dom_1 = np.mean(weighted_dominance_results[player_1]) if weighted_dominance_results[player_1] else 0.5
    dom_2 = np.mean(weighted_dominance_results[player_2]) if weighted_dominance_results[player_2] else 0.5

    # save pre-match, avoiding leakage
    dominance_form_diff.append(dom_1 - dom_2)

    # ratio of games won on this match
    games_ratio_1 = games_won_1 / games_total_1 if games_total_1 > 0 else 0.5
    games_ratio_2 = 1 - games_ratio_1

    weighted_dominance_results[player_1].append(games_ratio_1)
    weighted_dominance_results[player_2].append(games_ratio_2)

#### Head to head statistics

Head-to-head features count previous wins between the same two players, globally and on the current surface. Values are stored before updating the current match to avoid data leakage.

In [176]:
h2h_results = defaultdict(lambda: {"wins": 0, "losses": 0})
h2h_surface_results = defaultdict(lambda: {"wins": 0, "losses": 0})

h2h_diff = []
h2h_surface_diff = []


def update_h2h(player_1, player_2, surface, y):
    """Store pre-match head-to-head differences, then update matchup history."""
    key_1 = (player_1, player_2)
    key_2 = (player_2, player_1)

    surface_key_1 = (player_1, player_2, surface)
    surface_key_2 = (player_2, player_1, surface)

    # Store pre-match H2H difference to avoid data leakage.
    h2h_diff.append(
        h2h_results[key_1]["wins"] - h2h_results[key_1]["losses"]
    )

    h2h_surface_diff.append(
        h2h_surface_results[surface_key_1]["wins"]
        - h2h_surface_results[surface_key_1]["losses"]
    )

    # Update H2H only after computing the feature.
    if y == 1:
        h2h_results[key_1]["wins"] += 1
        h2h_results[key_2]["losses"] += 1

        h2h_surface_results[surface_key_1]["wins"] += 1
        h2h_surface_results[surface_key_2]["losses"] += 1
    else:
        h2h_results[key_1]["losses"] += 1
        h2h_results[key_2]["wins"] += 1

        h2h_surface_results[surface_key_1]["losses"] += 1
        h2h_surface_results[surface_key_2]["wins"] += 1

In [177]:
for row in df_prepared.itertuples(index=False):
    player_1, player_2 = row.Player_1, row.Player_2
    match_date, surface = row.Date, row.Surface
    y = row.y

    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    fatigue_diff.append(player_fatigue(player_1, match_date) - player_fatigue(player_2, match_date))
    update_recent_results(player_1, player_2, surface, y)
    update_elo_ratings(player_1, player_2, surface, y)

    games_won_P1 = row.Games_P1
    games_total = row.Games_P1 + row.Games_P2
    update_dominance_form(player_1, player_2, games_won_P1, games_total)
    update_h2h(player_1, player_2, surface, y)

    recent_matches[player_1].append(match_date)
    recent_matches[player_2].append(match_date)

df_prepared["Recent_Form_Diff"] = recent_form_diff

df_prepared["Dominance_Form_Diff"] = dominance_form_diff

df_prepared["Recent_Surface_Form_Diff"] = recent_surface_form_diff

df_prepared["Fatigue_Diff"] = fatigue_diff

# Elo values are already well behaved; clipping is unnecessary for trees and
# calculating limits on the complete dataset would leak test-period information.
df_prepared["Elo_Diff"] = elo_diff
df_prepared["Surface_Elo_Diff"] = surface_elo_diff

# --- H2H ---
df_prepared["H2H_Diff"] = h2h_diff
df_prepared["H2H_Surface_Diff"] = h2h_surface_diff


numerical_features += [
    "Recent_Form_Diff", "Recent_Surface_Form_Diff", "Dominance_Form_Diff",
    
    "Fatigue_Diff",

    "Elo_Diff", "Surface_Elo_Diff",

    "H2H_Diff", "H2H_Surface_Diff",
]

In [178]:
"""Print computed ELO ratings"""

# sort elo ratings and print
sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
# filter "Sinner J." surface ratigs
# for surface in ["Hard", "Clay", "Grass"]:
    # print(f"Sinner J. {surface} Elo: {surface_elo_ratings.get(('Sinner J.', surface), STARTING_ELO)}")

[('sinner j.', 2357.7501136226897),
 ('alcaraz c.', 2262.9900868552604),
 ('djokovic n.', 2162.9930036662863),
 ('federer r.', 2116.264620454963),
 ('zverev a.', 2114.6810444622274),
 ('nadal r.', 2098.821472224209),
 ('soderling r.', 2020.9324873069008),
 ('jodar r.', 2019.788308439037),
 ('fritz t.', 2007.4628708857238),
 ('fils a.', 2004.1451433814514),
 ('del potro j.m.', 1992.0565251777114),
 ('medvedev d.', 1973.9891665858058),
 ('auger aliassime f.', 1967.8878898580642),
 ('draper j.', 1964.3354615483804),
 ('musetti l.', 1947.4254029508954),
 ('de minaur a.', 1943.6591437765774),
 ('paul t.', 1930.0041326200476),
 ('ruud c.', 1926.8742300918418),
 ('bautista r.', 1924.943850413652),
 ('shelton b.', 1912.929211266756),
 ('rune h.', 1904.8446024992445),
 ('lehecka j.', 1898.1723058284808),
 ('fonseca j.', 1897.368101158479),
 ('cobolli f.', 1895.5371654935109),
 ('tiafoe f.', 1893.553400566985),
 ('korda s.', 1889.671381758454),
 ('rublev a.', 1881.830529147499),
 ('bublik a.', 1

#### Categorical features

We create an imputer to manage missing values, using a **most frequent** strategy and **One hot encoding** for low-cardinality categorical features to convert them into numerical ones. 

We manage high cardinality categorical features, such as **Tournament**, by keeping only the most frequent values and grouping the others into a single category called **Other**.

In [179]:
categorical_features = ["Court", "Surface", "Round", "Tournament", "Series"]

#### Splitting and save the dataset

Now we can split the dataset into training and test sets

In [180]:
debug_features = [
    "Date", "Player_1", "Player_2",
    "Odds_1", "Odds_2", # for baseline computation
    "Rank_1", "Rank_2"  # testing accuracy for players with best ranking
]
features = debug_features + numerical_features + categorical_features + ["y"]

df_prepared = df_prepared[features].copy()


And save them as new `xlsx` datasets, to be used in the next notebook for model training and evaluation.

In [181]:
from pathlib import Path
from sklearn.model_selection import train_test_split

TRAINING_PATH = Path("../data/training/tennis_training.xlsx")
TESTING_PATH = Path("../data/testing/tennis_testing.xlsx")

train_df, test_df = train_test_split(
    df_prepared,
    train_size=0.7,
    shuffle=False
)

TRAINING_PATH.parent.mkdir(parents=True, exist_ok=True)
train_df.to_excel(TRAINING_PATH, index=False)

TESTING_PATH.parent.mkdir(parents=True, exist_ok=True)
test_df.to_excel(TESTING_PATH, index=False)